# Libraries

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns


from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

In [2]:
# dispongo la carpeta con funciones e importo ternara. 
from pathlib import Path
import sys

# Obtiene la ruta del directorio actual y sube un nivel (.parent)
raiz_proyecto = Path().resolve().parent

# Agrega la ruta al sistema si no está ya incluida
if str(raiz_proyecto) not in sys.path:
  sys.path.append(str(raiz_proyecto))

# Importa tu librería o módulo
from funciones.ternaria import ternaria

# URLs y Constantes

In [3]:
BASE_URL = os.path.join('/mnt/', 'c')
BASE_DATA_URL = os.path.join('/mnt/', 'e')
COMP_URL = os.path.join(BASE_URL, 'Users', 'marco', 'Desktop', 'MASTER', 'MASTER', 'DMEyF', 'COMP1')
DATA_FOLDER = os.path.join(BASE_DATA_URL, 'DATASETS', 'DMEyF')
DATA_URL = os.path.join(DATA_FOLDER, 'competencia_01_crudo.csv')
DATA_TERNARIA_URL = os.path.join(DATA_FOLDER, 'competencia_01_ternaria.csv')

SEED = 230047

# UTILS

In [4]:
def van_dongen_normalized(contingency):
    n = contingency.values.sum()
    sum_max_rows = contingency.max(axis=1).sum()  # para cada label, max sobre clusters
    sum_max_cols = contingency.max(axis=0).sum()  # para cada cluster, max sobre labels
    max_row_total = contingency.sum(axis=1).max()
    max_col_total = contingency.sum(axis=0).max()
    
    vdn = (2*n - sum_max_rows - sum_max_cols) / (2*n - max_row_total - max_col_total)
    return vdn

# LEEMOS DATOS

In [5]:
df = pd.read_csv(
    DATA_TERNARIA_URL,
    dtype={'numero_de_cliente': 'int32', 'foto_mes': 'int32'}
)

/tmp/ipykernel_3150/3175415450.py:1: DtypeWarning: Columns (0: ternaria) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


# ANALISIS

In [6]:
mask_baja = df['ternaria'].str.contains('BAJA') 

## Clustering sobre varialbles con sparcity menor al 90%

In [45]:
proporcion_ceros = (
    df.loc[mask_baja].select_dtypes(include="number")
      .eq(0)
      .sum()
      .div(len(df.loc[mask_baja]))
      .sort_values(ascending=False)
)
sparse_columns_09 = []
non_sparse_columns_09 = []
sparse_columns_08 = []
non_sparse_columns_08 = []
sparse_columns_07 = []
non_sparse_columns_07 = []
sparse_columns_06 = []
non_sparse_columns_06 = []
sparse_columns_05 = []
non_sparse_columns_05 = []
for columna in proporcion_ceros.items():
    if columna[1]>.9:
        sparse_columns_09.append(columna[0])
    else :
        non_sparse_columns_09.append(columna[0])
    if columna[1]>.8:
        sparse_columns_08.append(columna[0])
    else :
        non_sparse_columns_08.append(columna[0])
    if columna[1]>.7:
        sparse_columns_07.append(columna[0])
    else :
        non_sparse_columns_07.append(columna[0])
    if columna[1]>.6:
        sparse_columns_06.append(columna[0])
    else :
        non_sparse_columns_06.append(columna[0])
    if columna[1]>.5:
        sparse_columns_05.append(columna[0])
    else :
        non_sparse_columns_05.append(columna[0])

54 70 76
101 85 79


## K-Means

In [86]:
# columnas que no tienen valor informativo
columnas_excluir = [
    "numero_de_cliente",
    "foto_mes",
    "ternaria"
]
# Siempre agrego n registros de no baja para ver si el clustering los agrupa separado de los demas. 
def get_random_continua(k, mask_baja):
    indices_random_continua = (
        df.loc[df["ternaria"] == "continua"]
        .sample(
            n=round(mask_baja.sum() / k),
            random_state=SEED
        )
        .index)
    mask_random_continua = df.index.isin(indices_random_continua)
    return mask_random_continua

sparsity_rates = [0.3, .4, .5,  .6, .7, .8 , .9]
ks = [3,4,5,6]
resultados = []
dfs_baja = {}
for rate in sparsity_rates:
    sparse_columns = []
    non_sparse_columns = []  
    for columna in proporcion_ceros.items():
        if columna[1]>rate:
            sparse_columns.append(columna[0])
        else :
            non_sparse_columns.append(columna[0])
    _columnas_excluir = columnas_excluir + [x for x in sparse_columns]
    for k in ks:
        # agrego registros q no son de baja, quiero ver si el clustering los distingue
        mask_random = get_random_continua(k,mask_baja)
        df_baja = df.loc[mask_baja|mask_random].copy()
        df_baja.fillna(0, inplace=True) #naively filll with 0s
        labels_reales = df_baja['ternaria']
        X = df_baja.drop(columns = _columnas_excluir).select_dtypes(include='number')
        X_escalado = StandardScaler().fit_transform(X)

        kmeans = KMeans(
        n_clusters=k+1,
        n_init=20,
        random_state=SEED
        )
        column_name = f'k-mean-{k}-spars-{rate*10:.0f}'
        df_baja[column_name] = kmeans.fit_predict(X_escalado)
        dfs_baja[(rate, k)] = df_baja.copy()
        labels = kmeans.labels_
        resultados.append({
            'column_name':column_name,
            "sparsity_rate": rate,
            "k_baja": k,
            "clusters_totales": k + 1,
            "tamanios_clusters": pd.Series(labels).value_counts().sort_index().to_dict(),
            "columnas_usadas": X.shape[1],
            "columnas_excluidas_sparse": len(sparse_columns),
            "inercia": kmeans.inertia_,                                                   # menor
            "iteraciones": kmeans.n_iter_,
            "silhouette": silhouette_score(X_escalado, labels),                          # mayor
            "davies_bouldin": davies_bouldin_score(X_escalado, labels),                   # menor
            "calinski_harabasz": calinski_harabasz_score(X_escalado, labels)              # mayor
        })
tabla_resultados = pd.DataFrame(resultados)

tabla_resultados.sort_values(
    "silhouette",
    ascending=False
)

,column_name,sparsity_rate,k_baja,clusters_totales,tamanios_clusters,columnas_usadas,columnas_excluidas_sparse,inercia,iteraciones,silhouette,davies_bouldin,calinski_harabasz
0,k-mean-3-spars-3,0.3,3,4,"{0: 7389, 1: 8, 2: 4463, 3: 367}",47,106,4.653150e+05,24,0.144006,1.970643,850.451709
4,k-mean-3-spars-4,0.4,3,4,"{0: 3520, 1: 5, 2: 831, 3: 7871}",53,100,5.267875e+05,31,0.139017,1.775065,843.166022
5,k-mean-4-spars-4,0.4,4,5,"{0: 3375, 1: 903, 2: 58, 3: 7121, 4: 5}",53,100,4.760372e+05,30,0.129672,1.807245,721.943837
2,k-mean-5-spars-3,0.3,5,6,"{0: 47, 1: 4191, 2: 7, 3: 523, 4: 6231, 5: 5}",47,106,3.920311e+05,35,0.123656,1.757273,640.486640
16,k-mean-3-spars-7,0.7,3,4,"{0: 1338, 1: 7433, 2: 3424, 3: 32}",77,76,7.719902e+05,21,0.119310,2.271560,765.450572
12,k-mean-3-spars-6,0.6,3,4,"{0: 3850, 1: 7613, 2: 4, 3: 760}",68,85,6.740155e+05,47,0.114362,1.897703,803.764522
17,k-mean-4-spars-7,0.7,4,5,"{0: 1157, 1: 3375, 2: 2, 3: 6924, 4: 4}",77,76,7.003009e+05,14,0.112868,1.671769,651.743151
8,k-mean-3-spars-5,0.5,3,4,"{0: 3430, 1: 6404, 2: 2220, 3: 173}",57,96,5.724814e+05,11,0.111468,2.549741,798.744656
18,k-mean-5-spars-7,0.7,5,6,"{0: 68, 1: 3292, 2: 2, 3: 6538, 4: 1100, 5: 4}",77,76,6.459895e+05,32,0.107758,1.665871,610.554162
13,k-mean-4-spars-6,0.6,4,5,"{0: 116, 1: 1640, 2: 3299, 3: 6403, 4: 4}",68,85,6.073532e+05,39,0.107691,2.119350,703.332464


In [87]:
pd.Series(labels).value_counts().sort_index().to_dict()

{0: 5400, 1: 2629, 2: 369, 3: 2, 4: 61, 5: 559, 6: 1678}

In [82]:
df_baja = dfs_baja[(0.3, 3)]
pd.crosstab(
        df_baja["k-mean-3-spars-3"],
        df_baja["ternaria"],
        margins=True
    )

ternaria,BAJA+1,BAJA+2,continua,All
k-mean-3-spars-3,,,,
0,3569,2760,1060,7389
1,2,1,5,8
2,1469,1254,1740,4463
3,63,52,252,367
All,5103,4067,3057,12227


## Hierarchical

In [ ]:
sparsity_rates = [0.3, .4, .5, .6, .7, .8, .9]
ks = [3, 4, 5, 6]
linkages = ["ward", "complete", "average", "single"]
metrics = ["euclidean", "manhattan", "cosine"]

resultados = []
dfs_baja = {}

for rate in sparsity_rates:
    sparse_columns = []
    non_sparse_columns = []

    for columna in proporcion_ceros.items():
        if columna[1] > rate:
            sparse_columns.append(columna[0])
        else:
            non_sparse_columns.append(columna[0])

    _columnas_excluir = columnas_excluir + [x for x in sparse_columns]

    for k in ks:
        mask_random = get_random_continua(k, mask_baja)
        df_baja = df.loc[mask_baja | mask_random].copy()
        df_baja.fillna(0, inplace=True)

        labels_reales = df_baja["ternaria"]

        X = (
            df_baja
            .drop(columns=_columnas_excluir)
            .select_dtypes(include="number")
        )

        X_escalado = StandardScaler().fit_transform(X)

        for linkage in linkages:
            for metric in metrics:

                # Ward solamente admite distancia euclídea
                if linkage == "ward" and metric != "euclidean":
                    continue

                modelo = AgglomerativeClustering(
                    n_clusters=k + 1,
                    linkage=linkage,
                    metric=metric
                )

                labels = modelo.fit_predict(X_escalado)

                column_name = (
                    f"agg-{k}-{linkage}-{metric}-spars-{rate*10:.0f}"
                )

                df_baja[column_name] = labels

                resultados.append({
                    "column_name": column_name,
                    "sparsity_rate": rate,
                    "k_baja": k,
                    "clusters_totales": k + 1,
                    "linkage": linkage,
                    "metric": metric,
                    "columnas_usadas": X.shape[1],
                    "columnas_excluidas_sparse": len(sparse_columns),
                    "silhouette": silhouette_score(X_escalado, labels),
                    "davies_bouldin": davies_bouldin_score(
                        X_escalado,
                        labels
                    ),
                    "calinski_harabasz": calinski_harabasz_score(
                        X_escalado,
                        labels
                    ),
                    "tamanios_clusters": (
                        pd.Series(labels)
                        .value_counts()
                        .sort_index()
                        .to_dict()
                    )
                })

        dfs_baja[(rate, k)] = df_baja.copy()

tabla_resultados = pd.DataFrame(resultados)

In [ ]:
#Guardamos resultados
# tabla_resultados.to_csv(os.path.join(os.getcwd(), 'clus_hier_sparse.csv' ))

In [111]:
tabla_resultados.sort_values(by='silhouette', ascending=False).iloc[155:175]

,column_name,sparsity_rate,k_baja,clusters_totales,linkage,metric,columnas_usadas,columnas_excluidas_sparse,silhouette,davies_bouldin,calinski_harabasz,tamanios_clusters
92,agg-4-complete-manhattan-spars-5,0.5,4,5,complete,manhattan,57,96,0.728739,1.027458,284.602129,"{0: 11393, 1: 65, 2: 1, 3: 1, 4: 2}"
112,agg-6-complete-manhattan-spars-5,0.5,6,7,complete,manhattan,57,96,0.721408,0.857263,271.544332,"{0: 10648, 1: 12, 2: 3, 3: 2, 4: 24, 5: 1, 6: 8}"
192,agg-6-complete-manhattan-spars-7,0.7,6,7,complete,manhattan,77,76,0.721055,1.047652,300.885112,"{0: 10635, 1: 12, 2: 32, 3: 3, 4: 2, 5: 1, 6: 13}"
232,agg-6-complete-manhattan-spars-8,0.8,6,7,complete,manhattan,83,70,0.715611,1.233796,272.832435,"{0: 10631, 1: 46, 2: 13, 3: 3, 4: 2, 5: 1, 6: 2}"
262,agg-5-complete-manhattan-spars-9,0.9,5,6,complete,manhattan,99,54,0.678950,1.388425,275.414828,"{0: 53, 1: 2, 2: 53, 3: 3, 4: 10892, 5: 1}"
72,agg-6-complete-manhattan-spars-4,0.4,6,7,complete,manhattan,53,100,0.649275,0.852970,235.046634,"{0: 10648, 1: 7, 2: 31, 3: 2, 4: 1, 5: 1, 6: 8}"
142,agg-5-complete-manhattan-spars-6,0.6,5,6,complete,manhattan,68,85,0.616629,1.051805,318.391674,"{0: 10913, 1: 13, 2: 71, 3: 4, 4: 2, 5: 1}"
62,agg-5-complete-manhattan-spars-4,0.4,5,6,complete,manhattan,53,100,0.515659,1.275989,291.223365,"{0: 359, 1: 7, 2: 10610, 3: 26, 4: 1, 5: 1}"
22,agg-5-complete-manhattan-spars-3,0.3,5,6,complete,manhattan,47,106,0.508618,0.868048,315.945207,"{0: 10806, 1: 184, 2: 2, 3: 2, 4: 9, 5: 1}"
12,agg-4-complete-manhattan-spars-3,0.3,4,5,complete,manhattan,47,106,0.491187,1.133248,384.620609,"{0: 276, 1: 8, 2: 1, 3: 11168, 4: 9}"


In [112]:
df_baja = dfs_baja[(0.5, 5)]
pd.crosstab(
        df_baja["agg-5-complete-manhattan-spars-5"],
        df_baja["ternaria"],
        margins=True
    )
# df_baja

ternaria,BAJA+1,BAJA+2,continua,All
agg-5-complete-manhattan-spars-5,,,,
0,1156,909,872,2937
1,4,2,2,8
2,3934,3148,948,8030
3,1,1,0,2
4,8,7,11,26
5,0,0,1,1
All,5103,4067,1834,11004
